In [4]:
import os
import time
import json
import re
import csv
import traceback
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import quote
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# 🧼 Sanitize filenames for Windows
def safe_filename(variant):
    return re.sub(r'[<>:"/\\|?*]', "_", variant)

# 🔁 Double-encode variant names for CFTR2 URLs
def double_encode(variant):
    return quote(quote(variant, safe=""), safe="")

# 📥 Load and normalize all variant names from Excel
df = pd.read_excel("CFTR2_25September2024.xlsx", sheet_name="CFTR2 variants by legacy name", skiprows=9)
variant_names = [v.strip() for v in df["Variant legacy name"].dropna().unique()]

# 💾 Load existing data if available
if os.path.exists("cftr2_sweat_chloride.json"):
    with open("cftr2_sweat_chloride.json", encoding="utf-8") as f:
        content = f.read().strip()
        if content:
            all_data = json.loads(content)
        else:
            all_data = {}
else:
    all_data = {}

SAVE_FOLDER = "cftr2_variants"
os.makedirs(SAVE_FOLDER, exist_ok=True)

# 🔧 Chrome setup
options = Options()
# options.add_argument("--headless=new")  # Uncomment when stable
options.add_argument("--start-maximized")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 10)

# 🧪 Extractor with correct autofill logic
def extract_sweat_chloride_from_table(filename):
    from bs4 import BeautifulSoup
    import re

    with open(filename, encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    result = {
        "sweat_chloride_variant": None,
        "n_variant": None,
        "sweat_chloride_two_cf": None,
        "n_two_cf": None,
        "average_age_variant": None,
        "average_age_two_cf": None
    }

    tables = soup.find_all("table")
    for table in tables:
        headers = table.find_all("thead")
        if not headers:
            continue
        header_text = headers[0].get_text()
        # 🔍 Debug preview (optional)
        print("🔍 Table header preview:", header_text.strip()[:100])

        # ✅ Match sweat chloride table by structure
        if "variant" in header_text.lower() and "number of patients" in header_text.lower():
            rows = table.find_all("tr")
            if len(rows) >= 3:
                # Header row with sample sizes
                header_cells = rows[0].find_all("td")
                if len(header_cells) >= 3:
                    match_variant = re.search(r"number of patients\s*=\s*([\d,]+)", header_cells[1].text)
                    match_two_cf = re.search(r"number of patients\s*=\s*([\d,]+)", header_cells[2].text)
                    if match_variant:
                        result["n_variant"] = int(match_variant.group(1).replace(",", ""))
                    if match_two_cf:
                        result["n_two_cf"] = int(match_two_cf.group(1).replace(",", ""))

                # Row 1: sweat chloride averages
                data_cells = rows[1].find_all("td")
                if len(data_cells) >= 3:
                    def parse_value(text):
                        return int(re.sub(r"[^\d]", "", text)) if re.search(r"\d", text) else None
                    result["sweat_chloride_variant"] = parse_value(data_cells[1].text)
                    result["sweat_chloride_two_cf"] = parse_value(data_cells[2].text)

                # Row 3: average age
                if len(rows) >= 4:
                    age_cells = rows[3].find_all("td")
                    if len(age_cells) >= 3:
                        def parse_age(text):
                            return int(re.sub(r"[^\d]", "", text)) if re.search(r"\d", text) else None
                        result["average_age_variant"] = parse_age(age_cells[1].text)
                        result["average_age_two_cf"] = parse_age(age_cells[2].text)

            break

    # ✅ Autofill defaults if missing
    if result["sweat_chloride_two_cf"] is None:
        result["sweat_chloride_two_cf"] = 95
    if result["average_age_two_cf"] is None:
        result["average_age_two_cf"] = 22

    return result


# 🛂 Accept CFTR2 agreement
print("🔐 Navigating to CFTR2 welcome page...")
driver.get("https://cftr2.org/welcome")
time.sleep(2)

checkbox_ids = ["edit-education", "edit-individual", "edit-discuss", "edit-privacy"]
for box_id in checkbox_ids:
    try:
        checkbox = wait.until(EC.presence_of_element_located((By.ID, box_id)))
        driver.execute_script("arguments[0].scrollIntoView(true);", checkbox)
        time.sleep(0.5)
        if not checkbox.is_selected():
            driver.execute_script("arguments[0].click();", checkbox)
        print(f"✅ Checked: {box_id}")
    except Exception as e:
        print(f"⚠️ Could not check box {box_id}: {e}")

try:
    submit_button = wait.until(EC.element_to_be_clickable((By.ID, "edit-submit-terms")))
    submit_button.click()
    print("✅ Submitted agreement form.")
except Exception as e:
    print(f"⚠️ Could not click submit button: {e}")

# 🌀 Process each variant
failed_variants = []
variant_log = []

for variant in variant_names:
    variant_clean = variant.strip()
    if variant_clean in (k.strip() for k in all_data):
        print(f"⏭️ Skipping already processed variant: {variant_clean}")
        continue

    try:
        print(f"\n🌐 Processing: {variant_clean}")
        encoded_variant = double_encode(variant_clean)
        url = f"https://cftr2.org/mutation/general/{encoded_variant}"
        print(f"🔗 Encoded URL: {url}")
        driver.get(url)

        if "The website encountered an unexpected error" in driver.page_source:
            print(f"⚠️ CFTR2 error page for variant: {variant_clean}")
            failed_variants.append(variant_clean)
            variant_log.append([variant_clean, url, "CFTR2 Error Page"])
            continue

        try:
            header = driver.find_element(By.CSS_SELECTOR, "h1").text.strip()
            if variant_clean.replace(" ", "") not in header.replace(" ", ""):
                print(f"⚠️ Mismatch: expected {variant_clean}, but page shows {header}")
        except Exception:
            print(f"⚠️ Could not verify variant name on page for {variant_clean}")

        sweat_tab = wait.until(EC.presence_of_element_located((By.XPATH, "//a[contains(text(), 'Sweat Chloride')]")))
        driver.execute_script("arguments[0].scrollIntoView(true);", sweat_tab)
        time.sleep(1)
        sweat_tab.click()

        wait.until(EC.text_to_be_present_in_element((By.TAG_NAME, "body"), "mEq/L"))

        safe_name = safe_filename(variant_clean)
        html_path = os.path.join(SAVE_FOLDER, f"{safe_name}.html")
        with open(html_path, "w", encoding="utf-8") as f:
            f.write(driver.page_source)
        print(f"✅ Saved: {html_path}")

        data = extract_sweat_chloride_from_table(html_path)
        all_data[variant_clean] = data
        print(f"🧪 Extracted: {data}")
        variant_log.append([variant_clean, url, "Success"])

        with open("cftr2_sweat_chloride.json", "w", encoding="utf-8") as f:
            json.dump(all_data, f, indent=2)

    except Exception as e:
        print(f"❌ Error with {variant_clean}: {e}")
        failed_variants.append(variant_clean)
        variant_log.append([variant_clean, url, f"Error: {str(e)}"])
        traceback.print_exc()

driver.quit()

# 📝 Save failed variants
with open("failed_variants.txt", "w") as f:
    f.write("\n".join(failed_variants))

# 📄 Save variant URL log
with open("variant_url_log.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Variant", "Encoded URL", "Status"])
    writer.writerows(variant_log)

print("\n📁 All data saved to cftr2_sweat_chloride.json")
print("📄 Failed variants saved to failed_variants.txt")
print("📊 Variant URL log saved to variant_url_log.csv")
print("✅ Script completed successfully!")

🔐 Navigating to CFTR2 welcome page...
✅ Checked: edit-education
✅ Checked: edit-individual
✅ Checked: edit-discuss
✅ Checked: edit-privacy
✅ Submitted agreement form.

🌐 Processing: F508del
🔗 Encoded URL: https://cftr2.org/mutation/general/F508del
⚠️ Mismatch: expected F508del, but page shows 
✅ Saved: cftr2_variants\F508del.html
🔍 Table header preview: Average in people who do not have CF
                    

                        Average of all pa
🧪 Extracted: {'sweat_chloride_variant': 96, 'n_variant': 91639, 'sweat_chloride_two_cf': 95, 'n_two_cf': 122935, 'average_age_variant': 22, 'average_age_two_cf': 22}

🌐 Processing: G542X
🔗 Encoded URL: https://cftr2.org/mutation/general/G542X
⚠️ Mismatch: expected G542X, but page shows 
✅ Saved: cftr2_variants\G542X.html
🔍 Table header preview: Average in people who do not have CF
                    

                        Average of all pa
🧪 Extracted: {'sweat_chloride_variant': 98, 'n_variant': 5496, 'sweat_chloride_two_cf': 95, 'n_

Traceback (most recent call last):
  File "C:\Users\adith\AppData\Local\Temp\ipykernel_14680\3565357261.py", line 176, in <module>
    wait.until(EC.text_to_be_present_in_element((By.TAG_NAME, "body"), "mEq/L"))
    ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\adith\AppData\Roaming\Python\Python313\site-packages\selenium\webdriver\support\wait.py", line 138, in until
    raise TimeoutException(message, screen, stacktrace)
selenium.common.exceptions.TimeoutException: Message: 



⚠️ Mismatch: expected E60X, but page shows 
✅ Saved: cftr2_variants\E60X.html
🔍 Table header preview: Average in people who do not have CF
                    

                        Average of all pa
🧪 Extracted: {'sweat_chloride_variant': 101, 'n_variant': 406, 'sweat_chloride_two_cf': 95, 'n_two_cf': 122935, 'average_age_variant': 24, 'average_age_two_cf': 22}

🌐 Processing: P67L
🔗 Encoded URL: https://cftr2.org/mutation/general/P67L
⚠️ Mismatch: expected P67L, but page shows 
✅ Saved: cftr2_variants\P67L.html
🔍 Table header preview: Average in people who do not have CF
                    

                        Average of all pa
🧪 Extracted: {'sweat_chloride_variant': 60, 'n_variant': 403, 'sweat_chloride_two_cf': 95, 'n_two_cf': 122935, 'average_age_variant': 30, 'average_age_two_cf': 22}

🌐 Processing: 1154insTC
🔗 Encoded URL: https://cftr2.org/mutation/general/1154insTC
⚠️ Mismatch: expected 1154insTC, but page shows 
✅ Saved: cftr2_variants\1154insTC.html
🔍 Table header pr

Traceback (most recent call last):
  File "C:\Users\adith\AppData\Local\Temp\ipykernel_14680\3565357261.py", line 176, in <module>
    wait.until(EC.text_to_be_present_in_element((By.TAG_NAME, "body"), "mEq/L"))
    ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\adith\AppData\Roaming\Python\Python313\site-packages\selenium\webdriver\support\wait.py", line 138, in until
    raise TimeoutException(message, screen, stacktrace)
selenium.common.exceptions.TimeoutException: Message: 



⚠️ Mismatch: expected R1066H, but page shows 
✅ Saved: cftr2_variants\R1066H.html
🔍 Table header preview: Average in people who do not have CF
                    

                        Average of all pa
🧪 Extracted: {'sweat_chloride_variant': 81, 'n_variant': 190, 'sweat_chloride_two_cf': 95, 'n_two_cf': 122935, 'average_age_variant': 25, 'average_age_two_cf': 22}

🌐 Processing: R352Q
🔗 Encoded URL: https://cftr2.org/mutation/general/R352Q
⚠️ Mismatch: expected R352Q, but page shows 
✅ Saved: cftr2_variants\R352Q.html
🔍 Table header preview: Average in people who do not have CF
                    

                        Average of all pa
🧪 Extracted: {'sweat_chloride_variant': 80, 'n_variant': 189, 'sweat_chloride_two_cf': 95, 'n_two_cf': 122935, 'average_age_variant': 26, 'average_age_two_cf': 22}

🌐 Processing: G1244E
🔗 Encoded URL: https://cftr2.org/mutation/general/G1244E
⚠️ Mismatch: expected G1244E, but page shows 
✅ Saved: cftr2_variants\G1244E.html
🔍 Table header preview

...

⚠️ Mismatch: expected c.4187delC, but page shows 
✅ Saved: cftr2_variants\c.4187delC.html
🔍 Table header preview: Average in people who do not have CF
                    

                        Average of all pa
🧪 Extracted: {'sweat_chloride_variant': None, 'n_variant': 1, 'sweat_chloride_two_cf': 95, 'n_two_cf': 122935, 'average_age_variant': None, 'average_age_two_cf': 22}

🌐 Processing: c.583delC
🔗 Encoded URL: https://cftr2.org/mutation/general/c.583delC
⚠️ Mismatch: expected c.583delC, but page shows 
❌ Error with c.583delC: Message: 
Stacktrace:
	GetHandleVerifier [0x00007FF6D3401522+60802]
	(No symbol) [0x00007FF6D337AC22]
	(No symbol) [0x00007FF6D3237CE4]
	(No symbol) [0x00007FF6D3286D4D]
	(No symbol) [0x00007FF6D3286E1C]
	(No symbol) [0x00007FF6D32CCE37]
	(No symbol) [0x00007FF6D32AABBF]
	(No symbol) [0x00007FF6D32CA224]
	(No symbol) [0x00007FF6D32AA923]
	(No symbol) [0x00007FF6D3278FEC]
	(No symbol) [0x00007FF6D3279C21]
	GetHandleVerifier [0x00007FF6D37041BD+3217949]
	GetH

Traceback (most recent call last):
  File "C:\Users\adith\AppData\Local\Temp\ipykernel_14680\3565357261.py", line 171, in <module>
    sweat_tab = wait.until(EC.presence_of_element_located((By.XPATH, "//a[contains(text(), 'Sweat Chloride')]")))
  File "C:\Users\adith\AppData\Roaming\Python\Python313\site-packages\selenium\webdriver\support\wait.py", line 138, in until
    raise TimeoutException(message, screen, stacktrace)
selenium.common.exceptions.TimeoutException: Message: 
Stacktrace:
	GetHandleVerifier [0x00007FF6D3401522+60802]
	(No symbol) [0x00007FF6D337AC22]
	(No symbol) [0x00007FF6D3237CE4]
	(No symbol) [0x00007FF6D3286D4D]
	(No symbol) [0x00007FF6D3286E1C]
	(No symbol) [0x00007FF6D32CCE37]
	(No symbol) [0x00007FF6D32AABBF]
	(No symbol) [0x00007FF6D32CA224]
	(No symbol) [0x00007FF6D32AA923]
	(No symbol) [0x00007FF6D3278FEC]
	(No symbol) [0x00007FF6D3279C21]
	GetHandleVerifier [0x00007FF6D37041BD+3217949]
	GetHandleVerifier [0x00007FF6D3746157+3488183]
	GetHandleVerifier [0x00

⚠️ Mismatch: expected c.88_89delCA, but page shows 
✅ Saved: cftr2_variants\c.88_89delCA.html
🔍 Table header preview: Average in people who do not have CF
                    

                        Average of all pa
🧪 Extracted: {'sweat_chloride_variant': None, 'n_variant': 1, 'sweat_chloride_two_cf': 95, 'n_two_cf': 122935, 'average_age_variant': None, 'average_age_two_cf': 22}

🌐 Processing: 1027delG
🔗 Encoded URL: https://cftr2.org/mutation/general/1027delG
⚠️ Mismatch: expected 1027delG, but page shows 
❌ Error with 1027delG: Message: 
Stacktrace:
	GetHandleVerifier [0x00007FF6D3401522+60802]
	(No symbol) [0x00007FF6D337AC22]
	(No symbol) [0x00007FF6D3237CE4]
	(No symbol) [0x00007FF6D3286D4D]
	(No symbol) [0x00007FF6D3286E1C]
	(No symbol) [0x00007FF6D32CCE37]
	(No symbol) [0x00007FF6D32AABBF]
	(No symbol) [0x00007FF6D32CA224]
	(No symbol) [0x00007FF6D32AA923]
	(No symbol) [0x00007FF6D3278FEC]
	(No symbol) [0x00007FF6D3279C21]
	GetHandleVerifier [0x00007FF6D37041BD+3217949]
	GetH

Traceback (most recent call last):
  File "C:\Users\adith\AppData\Local\Temp\ipykernel_14680\3565357261.py", line 171, in <module>
    sweat_tab = wait.until(EC.presence_of_element_located((By.XPATH, "//a[contains(text(), 'Sweat Chloride')]")))
  File "C:\Users\adith\AppData\Roaming\Python\Python313\site-packages\selenium\webdriver\support\wait.py", line 138, in until
    raise TimeoutException(message, screen, stacktrace)
selenium.common.exceptions.TimeoutException: Message: 
Stacktrace:
	GetHandleVerifier [0x00007FF6D3401522+60802]
	(No symbol) [0x00007FF6D337AC22]
	(No symbol) [0x00007FF6D3237CE4]
	(No symbol) [0x00007FF6D3286D4D]
	(No symbol) [0x00007FF6D3286E1C]
	(No symbol) [0x00007FF6D32CCE37]
	(No symbol) [0x00007FF6D32AABBF]
	(No symbol) [0x00007FF6D32CA224]
	(No symbol) [0x00007FF6D32AA923]
	(No symbol) [0x00007FF6D3278FEC]
	(No symbol) [0x00007FF6D3279C21]
	GetHandleVerifier [0x00007FF6D37041BD+3217949]
	GetHandleVerifier [0x00007FF6D3746157+3488183]
	GetHandleVerifier [0x00

⚠️ Mismatch: expected S531X, but page shows 
✅ Saved: cftr2_variants\S531X.html
🔍 Table header preview: Average in people who do not have CF
                    

                        Average of all pa
🧪 Extracted: {'sweat_chloride_variant': None, 'n_variant': 1, 'sweat_chloride_two_cf': 95, 'n_two_cf': 122935, 'average_age_variant': None, 'average_age_two_cf': 22}

🌐 Processing: 994del9
🔗 Encoded URL: https://cftr2.org/mutation/general/994del9
⚠️ Mismatch: expected 994del9, but page shows 
✅ Saved: cftr2_variants\994del9.html
🔍 Table header preview: Average in people who do not have CF
                    

                        Average of all pa
🧪 Extracted: {'sweat_chloride_variant': None, 'n_variant': 1, 'sweat_chloride_two_cf': 95, 'n_two_cf': 122935, 'average_age_variant': None, 'average_age_two_cf': 22}

🌐 Processing: 1710del17bp
🔗 Encoded URL: https://cftr2.org/mutation/general/1710del17bp
⚠️ Mismatch: expected 1710del17bp, but page shows 
✅ Saved: cftr2_variants\1710del17